# 26b · GSE135779 · scRNA_seq (pseudobulk) · scoring these samples on the GSE65391 modules

Reads this study's `expression.rds` and the GSE65391 end product `module_loadings.csv` (for each module:
its genes and their weights; no GSE65391 expression values). Writes `projected_scores.rds` in
`data/run_artifacts/GSE135779/`.

**What is transferred.** The model fitted on the GSE65391 first visits (notebook 05b): for each module,
the genes and their weights (u1 / d1 of the module's first principal component). Scoring a new data set
with a model learned elsewhere is transfer learning by projection (Stein-O'Brien GL et al. Decomposing
cell identity for transfer learning across cellular measurements, platforms, tissues, and species.
*Cell Syst* 2019;8:395-411, doi:10.1016/j.cels.2019.04.004; Sharma G et al. projectR. *Bioinformatics*
2020;36:3592-3593, doi:10.1093/bioinformatics/btaa183).

**Scaling is local.** GSE65391 standardised each gene with its own first-visit mean and SD. These are
array intensities; here the values are RNA sequencing log counts on another scale, so the array's means
and SDs cannot be applied. Each gene is standardised within this study, with the mean and SD of its 33
children with SLE (the GSE65391 modules were fitted on children with SLE), and then weighted with the
array weights. The 11 healthy children are scored on the same scale:

  score = sum over the module's genes measured here of z x weight

**Coverage.** Genes that this study does not measure are left out, and the share of each module's genes
used is reported.

**Check.** For each module, the projected score against the eigengene recomputed from this study's own
data, over the 33 children with SLE: Pearson r. A high r means the array's weighting and a local
refit agree on the module's axis in these samples.

## Standardize this study's genes and check coverage

In [1]:
source("../src/paths.R")
x  <- readRDS(art("GSE135779", "expression.rds"))
ld <- read.csv(art("GSE65391", "module_loadings.csv"))
E   <- x$E
sle <- x$meta$sle == 1
mu  <- rowMeans(E[, sle]); s <- apply(E[, sle], 1, sd)
Z   <- ((E - mu) / s)[s > 0, ]                           # genes x children, standardised on the SLE children
cov <- do.call(rbind, lapply(split(ld, ld$module), function(d)
  data.frame(module = d$module[1], genes_in_array_module = nrow(d), genes_here = sum(d$gene %in% rownames(Z)))))
cov$share_used <- round(cov$genes_here / cov$genes_in_array_module, 3)
cov[order(cov$share_used), ]

,module,genes_in_array_module,genes_here,share_used
,<chr>,<int>,<int>,<dbl>
cyan,cyan,60,34,0.567
yellow,yellow,383,261,0.681
green,green,315,241,0.765
brown,brown,647,517,0.799
magenta,magenta,180,153,0.850
blue,blue,1557,1347,0.865
pink,pink,212,184,0.868
tan,tan,124,109,0.879
black,black,280,254,0.907


**Explanation**
- As in notebook 15b, with one difference: each gene is standardized with the mean and SD of the 33
  children with SLE, and the 11 healthy children are scored on the same scale.

**Result.** Coverage ranges from 57% (cyan) to 99% (salmon); 12 of 14 modules have 75% or more. Cyan
(neutrophil granule) has 57%: several granule genes are too rarely counted in PBMC to pass the filter of
notebook 23.

## Score the children

In [2]:
scores <- sapply(split(ld, ld$module), function(d) {
  d <- d[d$gene %in% rownames(Z), ]
  colSums(Z[d$gene, , drop = FALSE] * d$weight)
})
dim(scores)

[1] 44 14

**Explanation**
- As in notebook 15b: one score per child and module, for all 44 children.

**Check.** Projected score against the locally recomputed eigengene of the same gene list.

## Check against a local refit

In [3]:
suppressMessages(library(WGCNA))
ld_here <- ld[ld$gene %in% rownames(Z), ]
ME_local <- moduleEigengenes(t(E[ld_here$gene, sle]), colors = ld_here$module)$eigengenes
check <- data.frame(module = colnames(scores),
                    r_projected_vs_local = sapply(colnames(scores), function(mo) cor(scores[sle, mo], ME_local[, paste0("ME", mo)])),
                    row.names = NULL)
check$r_projected_vs_local <- round(check$r_projected_vs_local, 3)
check[order(check$r_projected_vs_local), ]

,module,r_projected_vs_local
,<chr>,<dbl>
6,greenyellow,0.346
10,red,0.736
9,purple,0.746
3,brown,0.859
12,tan,0.880
13,turquoise,0.884
14,yellow,0.902
4,cyan,0.947
7,magenta,0.957


**Explanation**
- As in notebook 15b, over the 33 children with SLE.

**Result.** For 10 of 14 modules the projected score and the local refit agree at r >= 0.88, including
black (interferon, 0.985), pink (plasma cell, 0.998) and salmon (monocyte, 0.999). Four agree less:
greenyellow (0.346), red (0.736), purple (0.746) and brown (0.859); all four are not preserved here
(notebook 26).

## Save

In [4]:
saveRDS(list(scores = scores, coverage = cov, check = check), art("GSE135779", "projected_scores.rds"))

**Explanation**
- As in notebook 15b.